In [ ]:
!pip install -q pyarrow

In [ ]:
import os
import re
import ast
import json
import math
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
LOCAL_MAPPING_PATH = Path("artifacts/faiss/restaurant_index_mapping.parquet")
LOCAL_METADATA_PATH = Path("artifacts/faiss/metadata.json")

DRIVE_MAPPING_PATH = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet")
DRIVE_METADATA_PATH = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json")

if DRIVE_MAPPING_PATH.exists():
    MAPPING_PATH = DRIVE_MAPPING_PATH
    FAISS_METADATA_PATH = DRIVE_METADATA_PATH
elif LOCAL_MAPPING_PATH.exists():
    MAPPING_PATH = LOCAL_MAPPING_PATH
    FAISS_METADATA_PATH = LOCAL_METADATA_PATH
else:
    raise FileNotFoundError(
        "Nu am gasit restaurant_index_mapping.parquet. "
    )

OUTPUT_DIR = MAPPING_PATH.parent.parent / "query_parser"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Mapping path:", MAPPING_PATH)
print("FAISS metadata path:", FAISS_METADATA_PATH)
print("Output dir:", OUTPUT_DIR)

Mapping path: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
FAISS metadata path: /content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json
Output dir: /content/drive/MyDrive/tablewise/artifacts_new/query_parser


In [ ]:
df = pd.read_parquet(MAPPING_PATH)

print("Loaded mapping shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head(3))

Loaded mapping shape: (1033798, 31)
Columns: ['faiss_idx', 'restaurant_id', 'name', 'country', 'region', 'province', 'city', 'city_filled', 'city_filled_norm', 'city_source', 'address', 'latitude', 'longitude', 'rating', 'price_level', 'price_bucket', 'top_tags_text', 'top_tags_norm_list', 'meals_text', 'meals_norm_list', 'features_text', 'features_norm_list', 'special_diets_text', 'special_diets_norm_list', 'popularity_detailed', 'popularity_rank_num', 'popularity_total_num', 'popularity_score', 'profile_quality_score', 'short_profile', 'profile_text']


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."


In [ ]:
if FAISS_METADATA_PATH.exists():
    with open(FAISS_METADATA_PATH, "r", encoding="utf-8") as f:
        faiss_metadata = json.load(f)
else:
    faiss_metadata = {}

faiss_metadata

{'created_at': '2026-05-04T16:00:35.667690Z',
 'input_path': '/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet',
 'embedding_data_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'normalized_embeddings': True,
 'faiss_index_type': 'IndexFlatIP',
 'use_sample': False,
 'sample_size': None,
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'python_version': '3.12.13',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2'}

In [ ]:
required_cols = [
    "faiss_idx",
    "restaurant_id",
    "name",
    "country",
    "city_filled",
    "city_filled_norm",
    "price_bucket",
    "profile_text",
    "short_profile",
]

missing_required = [c for c in required_cols if c not in df.columns]

assert not missing_required, f"Lipsesc coloane obligatorii: {missing_required}"

assert df["faiss_idx"].is_unique, "faiss_idx nu este unic."
assert len(df) == df["faiss_idx"].nunique(), "Numarul de randuri nu corespunde cu faiss_idx unic."

if "num_restaurants" in faiss_metadata:
    assert len(df) == faiss_metadata["num_restaurants"], "Mapping-ul nu corespunde cu metadata din Notebookul 2."

print("Validari OK.")
print("Rows in mapping:", len(df))

Validări OK.
Rows in mapping: 1033798


In [ ]:
def strip_accents(text):
    if text is None or pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = strip_accents(str(x)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_value(x, default="Unknown"):
    if x is None or pd.isna(x):
        return default

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return default

    return s


def contains_phrase(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return False

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None


def phrase_positions(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return []

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return [m.start() for m in re.finditer(pattern, text_norm)]

In [ ]:
def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        values = x.tolist()
    elif isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()

        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = re.split(r"[,|;/]", s)
        else:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_s = safe_value(item, default="").strip()

        if not item_s:
            continue

        key = normalize_text(item_s)

        if key and key not in seen:
            cleaned.append(item_s)
            seen.add(key)

    return cleaned

In [ ]:
list_base_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
]


for base in list_base_cols:
    text_col = f"{base}_text"
    list_col = f"{base}_list"
    norm_list_col = f"{base}_norm_list"

    if list_col not in df.columns:
        if text_col in df.columns:
            df[list_col] = df[text_col].apply(parse_possible_list)
        else:
            df[list_col] = [[] for _ in range(len(df))]
    else:
        df[list_col] = df[list_col].apply(parse_possible_list)

    if norm_list_col not in df.columns:
        df[norm_list_col] = df[list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )
    else:
        df[norm_list_col] = df[norm_list_col].apply(parse_possible_list)
        df[norm_list_col] = df[norm_list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )

for col in [
    "top_tags_list",
    "top_tags_norm_list",
    "meals_list",
    "meals_norm_list",
    "features_list",
    "features_norm_list",
    "special_diets_list",
    "special_diets_norm_list",
]:
    if col in df.columns:
        print(col, "example:", df[col].iloc[0])

top_tags_list example: ['Cheap Eats', 'French']
top_tags_norm_list example: ['cheap eats', 'french']
meals_list example: ['Lunch', 'Dinner']
meals_norm_list example: ['lunch', 'dinner']
features_list example: ['Reservations', 'Seating', 'Wheelchair Accessible', 'Serves Alcohol', 'Accepts Credit Cards', 'Table Service']
features_norm_list example: ['reservations', 'seating', 'wheelchair accessible', 'serves alcohol', 'accepts credit cards', 'table service']
special_diets_list example: []
special_diets_norm_list example: []


In [ ]:

for col in ["country", "region", "province", "city", "city_filled"]:
    if col in df.columns:
        norm_col = f"{col}_norm"

        if norm_col not in df.columns:
            df[norm_col] = df[col].apply(normalize_text)
        else:
            df[norm_col] = df[norm_col].apply(normalize_text)

df["city_filled_norm"] = df["city_filled"].apply(normalize_text)
df["country_norm"] = df["country"].apply(normalize_text)

display(df[["country", "country_norm", "city_filled", "city_filled_norm"]].head())

,country,country_norm,city_filled,city_filled_norm
0,France,france,Saint-Jouvent,saint-jouvent
1,France,france,Saint-Jouvent,saint-jouvent
2,France,france,Rivarennes,rivarennes
3,France,france,Lacelle,lacelle
4,France,france,Saint-Laurent-de-Levezou,saint-laurent-de-levezou


In [ ]:
BAD_CITY_TERMS = {
    "",
    "nan",
    "none",
    "null",
    "unknown",
    "street",
    "st",
    "road",
    "rd",
    "avenue",
    "ave",
    "square",
    "center",
    "centre",
    "park",
    "hotel",
    "restaurant",
    "restaurants",
    "best",
    "cheap",
    "budget",
    "expensive",
    "food",
    "pub",
    "bar",
    "cafe",
    "cafes",
    "coffee",
    "pizza",
    "pasta",
    "sushi",
    "seafood",
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "europe",
    "united kingdom uk",
    "united kingdom",
    "uk",
    "england",
    "scotland",
    "wales",
    "northern ireland",
    "france",
    "italy",
    "spain",
    "germany",
    "portugal",
    "greece",
}

In [ ]:
country_vocab = (
    df["country_norm"]
    .dropna()
    .astype(str)
    .map(normalize_text)
)

country_vocab = sorted(
    {c for c in country_vocab if c and c not in {"unknown", "nan", "none", "null"}},
    key=len,
    reverse=True,
)

COUNTRY_ALIASES = {
    "uk": "united kingdom",
    "u k": "united kingdom",
    "great britain": "united kingdom",
    "britain": "united kingdom",
    "england": "united kingdom",
    "scotland": "united kingdom",
    "wales": "united kingdom",
    "northern ireland": "united kingdom",
    "deutschland": "germany",
    "espana": "spain",
    "españa": "spain",
    "italia": "italy",
    "grecia": "greece",
}

print("Număr țări:", len(country_vocab))
print(country_vocab[:30])

Număr țări: 24
['northern ireland', 'the netherlands', 'czech republic', 'scotland', 'bulgaria', 'slovakia', 'portugal', 'england', 'belgium', 'ireland', 'romania', 'germany', 'hungary', 'finland', 'croatia', 'austria', 'denmark', 'poland', 'sweden', 'france', 'greece', 'wales', 'italy', 'spain']


In [ ]:
city_counts = df["city_filled_norm"].value_counts(dropna=True).to_dict()

country_values = set(df["country_norm"].dropna().astype(str).map(normalize_text))
region_values = set(df["region_norm"].dropna().astype(str).map(normalize_text)) if "region_norm" in df.columns else set()
province_values = set(df["province_norm"].dropna().astype(str).map(normalize_text)) if "province_norm" in df.columns else set()

blocked_geo_values = country_values | region_values | province_values | BAD_CITY_TERMS

raw_city_values = (
    df["city_filled_norm"]
    .dropna()
    .astype(str)
    .map(normalize_text)
    .tolist()
)

city_vocab = []

for city in sorted(set(raw_city_values), key=len, reverse=True):
    if not city:
        continue
    if city in blocked_geo_values:
        continue
    if len(city) < 2:
        continue
    if city.isdigit():
        continue

    if len(city.split()) > 6:
        continue

    city_vocab.append(city)

IMPORTANT_CITY_ALIASES = {
    "rome": ["rome", "roma"],
    "paris": ["paris"],
    "madrid": ["madrid"],
    "barcelona": ["barcelona"],
    "milan": ["milan", "milano"],
    "berlin": ["berlin"],
    "lisbon": ["lisbon", "lisboa"],
    "athens": ["athens", "athina", "athena"],
    "manchester": ["manchester"],
    "dublin": ["dublin"],
    "vienna": ["vienna", "wien"],
    "prague": ["prague", "praha"],
    "amsterdam": ["amsterdam"],
    "brussels": ["brussels", "bruxelles", "brussel"],
    "budapest": ["budapest"],
    "warsaw": ["warsaw", "warszawa"],
    "krakow": ["krakow", "kraków"],
    "porto": ["porto", "oporto"],
    "florence": ["florence", "firenze"],
    "venice": ["venice", "venezia"],
    "naples": ["naples", "napoli"],
    "turin": ["turin", "torino"],
    "seville": ["seville", "sevilla"],
    "valencia": ["valencia"],
    "granada": ["granada"],
    "malaga": ["malaga", "málaga"],
    "nice": ["nice"],
    "lyon": ["lyon"],
    "marseille": ["marseille"],
    "bordeaux": ["bordeaux"],
    "hamburg": ["hamburg"],
    "munich": ["munich", "muenchen", "münchen"],
    "cologne": ["cologne", "koln", "köln"],
    "frankfurt": ["frankfurt"],
}

important_canonical_cities = set(IMPORTANT_CITY_ALIASES.keys())

for canonical, aliases in IMPORTANT_CITY_ALIASES.items():
    canonical_norm = normalize_text(canonical)
    alias_norms = [normalize_text(a) for a in aliases]

    appears_in_data = any(a in city_counts for a in alias_norms)

    if appears_in_data or canonical_norm in important_canonical_cities:
        if canonical_norm not in city_vocab:
            city_vocab.append(canonical_norm)

important_order = list(IMPORTANT_CITY_ALIASES.keys())
city_vocab = sorted(
    set(city_vocab),
    key=lambda c: (
        0 if c in important_order else 1,
        important_order.index(c) if c in important_order else 999999,
        -len(c),
    ),
)

CITY_ALIASES = {}

for canonical, aliases in IMPORTANT_CITY_ALIASES.items():
    for alias in aliases:
        alias_norm = normalize_text(alias)
        CITY_ALIASES[alias_norm] = normalize_text(canonical)

print("Număr orașe safe:", len(city_vocab))
print("Top city counts:")
display(pd.Series(city_counts).sort_values(ascending=False).head(30))

print("\nSample city_vocab:")
print(city_vocab[:80])

print("\nVerificări:")
for c in ["rome", "paris", "madrid", "barcelona", "milan", "berlin", "lisbon", "athens", "london", "best", "street", "united kingdom"]:
    print(c, "->", c in city_vocab)

Număr orașe safe: 62249
Top city counts:


,0
paris,18126
rome,12603
madrid,12133
barcelona,10283
milan,8381
prague,6035
lisbon,5261
vienna,4571
amsterdam,4325
valencia,4100



Sample city_vocab:
['rome', 'paris', 'madrid', 'barcelona', 'milan', 'berlin', 'lisbon', 'athens', 'manchester', 'dublin', 'vienna', 'prague', 'amsterdam', 'brussels', 'budapest', 'warsaw', 'krakow', 'porto', 'florence', 'venice', 'naples', 'turin', 'seville', 'valencia', 'granada', 'malaga', 'nice', 'lyon', 'marseille', 'bordeaux', 'hamburg', 'munich', 'cologne', 'frankfurt', 'communaute d agglomeration du pays ajaccien', 'saint-martin-de-bienfaite-la-cressonniere', 'la vacquerie-et-saint-martin-de-castries', 'montigny mornay villeneuve sur vingeanne', 'communaute d agglomeration pau-pyrenees', 'saint-quentin-la-motte-croix-au-bailly', 'javerlhac-et-la-chapelle-saint-robert', 'passo della collina - collina vecchia', 'ordesa y monte perdido national park', 'zona artigianale localita campomorto', 'sankt michael in der obersteiermark', 'sao bartolomeu dos galegos e moledo', 'durfort-et-saint-martin-de-sossenac', 'angoustrine-villeneuve-des-escaldes', 'saint-felix-de-reillac-et-mortemart

In [ ]:
def flatten_norm_list_column(series):
    values = Counter()

    for xs in series:
        if not isinstance(xs, list):
            continue

        for x in xs:
            x_norm = normalize_text(x)

            if x_norm:
                values[x_norm] += 1

    return values


tag_counter = flatten_norm_list_column(df["top_tags_norm_list"]) if "top_tags_norm_list" in df.columns else Counter()
meal_counter = flatten_norm_list_column(df["meals_norm_list"]) if "meals_norm_list" in df.columns else Counter()
feature_counter = flatten_norm_list_column(df["features_norm_list"]) if "features_norm_list" in df.columns else Counter()
diet_counter = flatten_norm_list_column(df["special_diets_norm_list"]) if "special_diets_norm_list" in df.columns else Counter()

tag_vocab = sorted(tag_counter.keys(), key=len, reverse=True)
meal_vocab = sorted(meal_counter.keys(), key=len, reverse=True)
feature_vocab = sorted(feature_counter.keys(), key=len, reverse=True)
diet_vocab = sorted(diet_counter.keys(), key=len, reverse=True)

print("Tag vocab:", len(tag_vocab))
print("Meal vocab:", len(meal_vocab))
print("Feature vocab:", len(feature_vocab))
print("Diet vocab:", len(diet_vocab))

print("\nTop tags:")
display(pd.Series(tag_counter).sort_values(ascending=False).head(30))

print("\nTop meals:")
display(pd.Series(meal_counter).sort_values(ascending=False).head(30))

print("\nTop features:")
display(pd.Series(feature_counter).sort_values(ascending=False).head(30))

print("\nTop diets:")
display(pd.Series(diet_counter).sort_values(ascending=False).head(30))

Tag vocab: 189
Meal vocab: 6
Feature vocab: 39
Diet vocab: 5

Top tags:


,0
mid-range,515029
italian,231472
cheap eats,229404
european,170832
mediterranean,151224
vegetarian friendly,125504
pizza,110138
cafe,99934
french,97358
bar,85943



Top meals:


,0
dinner,510115
lunch,488298
breakfast,172513
drinks,112337
brunch,95731
after-hours,87039



Top features:


,0
seating,219991
reservations,208539
table service,186075
wheelchair accessible,142025
serves alcohol,124884
takeout,90749
outdoor seating,72331
accepts credit cards,55467
highchairs available,50778
full bar,46653



Top diets:


,0
vegetarian friendly,306424
vegan options,127448
gluten free options,116841
halal,5397
kosher,249


In [ ]:
PRICE_KEYWORDS = {
    "cheap": [
        "cheap",
        "budget",
        "affordable",
        "low cost",
        "low-cost",
        "inexpensive",
        "not expensive",
        "economic",
        "economical",
        "ieftin",
        "ieftina",
        "ieftine",
    ],
    "mid": [
        "mid range",
        "mid-range",
        "moderate",
        "casual",
        "normal price",
        "average price",
        "medium price",
        "mediu",
        "pret mediu",
        "preț mediu",
    ],
    "expensive": [
        "expensive",
        "fine dining",
        "luxury",
        "high end",
        "high-end",
        "premium",
        "fancy",
        "upscale",
        "scump",
        "scumpa",
        "scumpe",
        "lux",
    ],
}

TAG_SYNONYMS = {
    "italian": ["italian", "italian food", "pizza", "pasta", "trattoria", "osteria"],
    "french": ["french", "french food", "bistro", "brasserie"],
    "spanish": ["spanish", "spanish food", "tapas", "paella"],
    "greek": ["greek", "greek food", "taverna", "souvlaki", "gyros"],
    "portuguese": ["portuguese", "portuguese food"],
    "german": ["german", "german food"],
    "seafood": ["seafood", "fish", "oyster", "sushi"],
    "steakhouse": ["steakhouse", "steak", "grill"],
    "asian": ["asian", "chinese", "japanese", "thai", "vietnamese", "korean"],
    "indian": ["indian", "curry"],
    "mexican": ["mexican", "tacos", "burrito"],
    "mediterranean": ["mediterranean"],
    "fast food": ["fast food", "burger", "burgers", "kebab"],
    "coffee": ["coffee", "cafe", "cafes", "café"],
    "bar": ["bar", "pub", "drinks", "cocktails"],
}

DIETARY_SYNONYMS = {
    "vegetarian friendly": ["vegetarian", "veggie", "vegetarian friendly", "fara carne", "fără carne"],
    "vegan options": ["vegan", "vegan options", "plant based", "plant-based"],
    "gluten free options": ["gluten free", "gluten-free", "without gluten", "fara gluten", "fără gluten"],
}

MEAL_SYNONYMS = {
    "breakfast": ["breakfast", "mic dejun"],
    "brunch": ["brunch"],
    "lunch": ["lunch", "pranz", "prânz"],
    "dinner": ["dinner", "cina", "cină"],
    "drinks": ["drinks", "cocktails", "beer", "wine"],
}

FEATURE_SYNONYMS = {
    "reservations": ["reservation", "reservations", "booking", "book a table", "rezervare"],
    "outdoor seating": ["outdoor", "terrace", "terrace seating", "outdoor seating", "terasă", "terasa"],
    "delivery": ["delivery", "takeaway", "take out", "takeout"],
    "wheelchair accessible": ["wheelchair", "accessible", "wheelchair accessible"],
    "family friendly": ["family friendly", "kid friendly", "kids", "children", "familie", "copii"],
    "romantic": ["romantic", "date night", "cozy", "cosy"],
}

In [ ]:
def find_synonym_matches(query_norm, synonym_dict):
    found = []

    for canonical, synonyms in synonym_dict.items():
        for syn in synonyms:
            syn_norm = normalize_text(syn)

            if contains_phrase(query_norm, syn_norm):
                found.append(canonical)
                break

    return sorted(set(found))


def extract_price_bucket(query_norm):
    for bucket, keywords in PRICE_KEYWORDS.items():
        for kw in keywords:
            kw_norm = normalize_text(kw)

            if contains_phrase(query_norm, kw_norm):
                return bucket

    return None


def extract_vocab_matches(query_norm, vocab, max_matches=8, min_len=3):
    found = []

    for term in vocab:
        term_norm = normalize_text(term)

        if len(term_norm) < min_len:
            continue

        if contains_phrase(query_norm, term_norm):
            found.append(term_norm)

        if len(found) >= max_matches:
            break

    return sorted(set(found))

In [ ]:
def extract_country(query_norm):
    for alias, canonical in COUNTRY_ALIASES.items():
        alias_norm = normalize_text(alias)

        if contains_phrase(query_norm, alias_norm):
            canonical_norm = normalize_text(canonical)

            if canonical_norm in country_vocab:
                return canonical_norm

    for country in country_vocab:
        if contains_phrase(query_norm, country):
            return country

    return None

In [ ]:
LOCATION_PREPOSITIONS = {
    "in",
    "near",
    "around",
    "from",
    "at",
    "within",
    "inside",
    "langa",
    "lângă",
    "in apropiere de",
    "în",
}


def city_has_location_context(query_norm, city_norm):
    if not contains_phrase(query_norm, city_norm):
        return False

    prep_patterns = [
        rf"\bin\s+{re.escape(city_norm)}\b",
        rf"\bnear\s+{re.escape(city_norm)}\b",
        rf"\baround\s+{re.escape(city_norm)}\b",
        rf"\bat\s+{re.escape(city_norm)}\b",
        rf"\bfrom\s+{re.escape(city_norm)}\b",
        rf"\bin apropiere de\s+{re.escape(city_norm)}\b",
        rf"\blanga\s+{re.escape(city_norm)}\b",
        rf"\blângă\s+{re.escape(city_norm)}\b",
        rf"\bin\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orașul\s+{re.escape(city_norm)}\b",
    ]

    for pattern in prep_patterns:
        if re.search(pattern, query_norm):
            return True
    if len(city_norm) >= 4 and city_norm not in BAD_CITY_TERMS:
        return True

    return False


def extract_city(query_norm):
    for alias, canonical in CITY_ALIASES.items():
        alias_norm = normalize_text(alias)
        canonical_norm = normalize_text(canonical)

        if contains_phrase(query_norm, alias_norm) and canonical_norm in city_vocab:
            return canonical_norm

    candidates = []

    for city in city_vocab:
        if city_has_location_context(query_norm, city):
            candidates.append(city)

    if not candidates:
        return None

    return max(candidates, key=len)

In [ ]:
def parse_user_query(query):
    query_norm = normalize_text(query)

    city = extract_city(query_norm)
    country = extract_country(query_norm)
    price_bucket = extract_price_bucket(query_norm)

    tags_from_synonyms = find_synonym_matches(query_norm, TAG_SYNONYMS)
    dietary_from_synonyms = find_synonym_matches(query_norm, DIETARY_SYNONYMS)
    meals_from_synonyms = find_synonym_matches(query_norm, MEAL_SYNONYMS)
    features_from_synonyms = find_synonym_matches(query_norm, FEATURE_SYNONYMS)

    tags_from_vocab = extract_vocab_matches(query_norm, tag_vocab, max_matches=8)
    meals_from_vocab = extract_vocab_matches(query_norm, meal_vocab, max_matches=5)
    features_from_vocab = extract_vocab_matches(query_norm, feature_vocab, max_matches=5)
    diets_from_vocab = extract_vocab_matches(query_norm, diet_vocab, max_matches=5)

    tags = sorted(set(tags_from_synonyms + tags_from_vocab))
    matched_meals = sorted(set(meals_from_synonyms + meals_from_vocab))
    matched_features = sorted(set(features_from_synonyms + features_from_vocab))
    dietary = sorted(set(dietary_from_synonyms + diets_from_vocab))

    parsed = {
        "original_query": query,
        "normalized_query": query_norm,
        "city": city,
        "country": country,
        "price_bucket": price_bucket,
        "tags": tags,
        "dietary": dietary,
        "matched_meals": matched_meals,
        "matched_features": matched_features,
    }

    return parsed

In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "best cheap restaurant in northern ireland",
    "vegan restaurant near berlin with outdoor seating",
    "tapas place in madrid",
    "restaurant in United Kingdom",
    "restaurant in London",
]

for q in test_queries:
    print("=" * 120)
    print(q)
    print(json.dumps(parse_user_query(q), indent=2, ensure_ascii=False))

cheap italian restaurant in rome
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}
vegetarian brunch in barcelona
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}
romantic dinner in paris
{
  "original_query": "romantic dinner in paris",
  "normalized_query": "romantic dinner in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}
family friendly seafood place in lisbon
{
  "origina

In [ ]:
def list_contains_term(row_list, term):
    if not isinstance(row_list, list):
        return False

    term_norm = normalize_text(term)

    if not term_norm:
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if term_norm == item_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False


def list_contains_any(row_list, terms):
    return any(list_contains_term(row_list, term) for term in terms)

In [ ]:
def apply_location_filter(df_in, parsed_query, min_results=20, verbose=True):
    current = df_in.copy()
    before = len(current)

    city = parsed_query.get("city")
    country = parsed_query.get("country")

    if city:
        filtered = current[current["city_filled_norm"] == city].copy()

        if verbose:
            print(f"City filter city={city}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print(
                    "Orasul a fost detectat, dar nu exista in datasetul indexat "
                    "Returnam subset gol ca sa evităm recomandari gresite"
                )
            return filtered

        current = filtered

    if country:
        before_country = len(current)
        filtered = current[current["country_norm"] == country].copy()

        if verbose:
            print(f"Country filter country={country}: {before_country} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print(
                    "tara/regiunea a fost detectata, dar nu exista rezultate "
                    "Returnam subset gol ca sa evitam recomandari gresite"
                )
            return filtered

        current = filtered

    return current

In [ ]:
def apply_price_filter(df_in, parsed_query, min_results=20, verbose=True):
    current = df_in.copy()

    price_bucket = parsed_query.get("price_bucket")

    if not price_bucket:
        return current

    if "price_bucket" not in current.columns:
        return current

    before = len(current)
    filtered = current[current["price_bucket"] == price_bucket].copy()

    if verbose:
        print(f"Price filter price_bucket={price_bucket}: {before} -> {len(filtered)}")

    if len(filtered) >= min_results:
        return filtered

    if len(filtered) > 0 and before <= min_results:
        return filtered

    if verbose:
        print("Price filter prea strict. il pastram pentru metadata_score, nu ca filtru hard.")

    return current

In [ ]:
def compute_metadata_score_for_row(row, parsed_query):
    score = 0.0
    matched_reasons = []

    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    # Price
    if price_bucket and row.get("price_bucket") == price_bucket:
        score += 2.0
        matched_reasons.append(f"price={price_bucket}")

    # Tags / cuisines
    row_tags = row.get("top_tags_norm_list", [])

    for tag in tags:
        if list_contains_term(row_tags, tag):
            score += 2.0
            matched_reasons.append(f"tag={tag}")

    # Dietary
    row_diets = row.get("special_diets_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for diet in dietary:
        if list_contains_term(row_diets, diet) or list_contains_term(row_tags, diet) or list_contains_term(row_features, diet):
            score += 2.0
            matched_reasons.append(f"diet={diet}")

    # Meals
    row_meals = row.get("meals_norm_list", [])

    for meal in meals:
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            score += 1.0
            matched_reasons.append(f"meal={meal}")

    # Features
    for feature in features:
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            score += 1.0
            matched_reasons.append(f"feature={feature}")

    return score, sorted(set(matched_reasons))

In [ ]:
def add_metadata_scores(df_in, parsed_query):
    current = df_in.copy()

    if len(current) == 0:
        current["metadata_score"] = []
        current["metadata_match_reasons"] = []
        return current

    scores_and_reasons = current.apply(
        lambda row: compute_metadata_score_for_row(row, parsed_query),
        axis=1,
    )

    current["metadata_score"] = scores_and_reasons.apply(lambda x: x[0])
    current["metadata_match_reasons"] = scores_and_reasons.apply(lambda x: x[1])

    return current

In [ ]:
def apply_metadata_filter(df_in, parsed_query, min_results=20, verbose=True):
    current = add_metadata_scores(df_in, parsed_query)

    query_has_metadata_constraints = bool(
        parsed_query.get("price_bucket")
        or parsed_query.get("tags")
        or parsed_query.get("dietary")
        or parsed_query.get("matched_meals")
        or parsed_query.get("matched_features")
    )

    if not query_has_metadata_constraints:
        if verbose:
            print("No metadata constraints found.")
        return current

    before = len(current)

    filtered = current[current["metadata_score"] > 0].copy()

    if verbose:
        print(f"Metadata score filter > 0: {before} -> {len(filtered)}")

    if len(filtered) >= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if 0 < len(filtered) < min_results and before <= 100:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if verbose:
        print("Metadata filter prea strict. Păstrăm toate rezultatele, dar cu metadata_score calculat.")

    return current.sort_values("metadata_score", ascending=False).reset_index(drop=True)

In [ ]:
def parse_and_filter(query, df_in, min_results=20, verbose=True):
    parsed = parse_user_query(query)

    if verbose:
        print("Parsed query:")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("\nInitial rows:", len(df_in))

    current = df_in.copy()

    current = apply_location_filter(
        current,
        parsed,
        min_results=min_results,
        verbose=verbose,
    )

    if verbose:
        print("After location filter:", len(current))

    current = apply_price_filter(
        current,
        parsed,
        min_results=min_results,
        verbose=verbose,
    )

    if verbose:
        print("After price filter:", len(current))

    current = apply_metadata_filter(
        current,
        parsed,
        min_results=min_results,
        verbose=verbose,
    )

    if verbose:
        print("After metadata filter/scoring:", len(current))

    return parsed, current.reset_index(drop=True)

In [ ]:
def preview_filtered_results(df_in, n=10):
    cols = [
        "faiss_idx",
        "restaurant_id",
        "name",
        "city_filled",
        "country",
        "rating",
        "price_bucket",
        "metadata_score",
        "metadata_match_reasons",
        "top_tags_text",
        "special_diets_text",
        "meals_text",
        "features_text",
        "short_profile",
    ]

    cols = [c for c in cols if c in df_in.columns]

    return df_in[cols].head(n)

In [ ]:
query = "cheap italian restaurant in rome"

parsed, filtered_df = parse_and_filter(
    query,
    df,
    min_results=20,
    verbose=True,
)

preview_filtered_results(filtered_df, n=20)

Parsed query:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Price filter price_bucket=cheap: 12603 -> 2755
After price filter: 2755
Metadata score filter > 0: 2755 -> 2755
After metadata filter/scoring: 2755


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,663015,g187791-d10043473,Forno Dolce Forno,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Bakeries, Italian",Unknown,Brunch,"Takeout, Wheelchair Accessible","Forno Dolce Forno | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Bakeries, Italian"
1,663012,g187791-d10042395,I Nuovi Gladiatori,Rome,Italy,3.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Takeout, Delivery, Reservations, Wheelchair Accessible","I Nuovi Gladiatori | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
2,675525,g187791-d9789769,Bar della Paglia,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, European, Vegetarian Friendly",Vegetarian Friendly,Dinner,Unknown,"Bar della Paglia | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, European, Vegetarian Friendly"
3,675441,g187791-d9712051,Pasticceria Siciliana Roma,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Dessert, Italian, Cafe",Unknown,Breakfast,"Takeout, Wheelchair Accessible, Seating","Pasticceria Siciliana Roma | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Dessert, Italian, Cafe"
4,675471,g187791-d9735886,Harbour Cafe,Rome,Italy,5.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian",Unknown,"Brunch, Drinks, Breakfast, Lunch","Takeout, Outdoor Seating, Seating, Parking Available, Street Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Wine and Beer, Accepts Mastercard, Accepts Visa, Accepts Credit Cards","Harbour Cafe | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian"
5,675478,g187791-d9743204,Pizza e Coffee,Rome,Italy,4.5,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza",Unknown,Unknown,Reservations,"Pizza e Coffee | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza"
6,675494,g187791-d9752043,La Pesceria,Rome,Italy,4.5,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Mediterranean, Gluten Free Options",Gluten Free Options,"Lunch, Dinner",Unknown,"La Pesceria | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Mediterranean, Gluten Free Options"
7,675506,g187791-d9768131,Burger King,Rome,Italy,3.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Quick Bites, Italian, American",Unknown,Unknown,"Takeout, Wheelchair Accessible, Seating","Burger King | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Quick Bites, Italian, American"
8,675517,g187791-d9784422,Kalamaro Piadinaro,Rome,Italy,4.5,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Fast food, Mediterranean",Vegetarian Friendly,"Dinner, Lunch",Unknown,"Kalamaro Piadinaro | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Fast food, Mediterranean"
9,675521,g187791-d9788403,Trieste Pizza,Rome,Italy,4.5,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Fast food",Unknown,Unknown,Unknown,"Trieste Pizza | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Fast food"


In [ ]:
query = "vegetarian brunch in barcelona"

parsed, filtered_df = parse_and_filter(
    query,
    df,
    min_results=20,
    verbose=True,
)

preview_filtered_results(filtered_df, n=20)

Parsed query:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
After price filter: 10283
Metadata score filter > 0: 10283 -> 3713
After metadata filter/scoring: 3713


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,379304,g187497-d2467666,Bar del Pi,Barcelona,Spain,4.0,cheap,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Bar, Cafe, Mediterranean",Vegetarian Friendly,"Lunch, Dinner, Brunch, Drinks",Unknown,"Bar del Pi | Barcelona, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Bar, Cafe, Mediterranean"
1,382885,g187497-d8448147,Creps Barcelona,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Cafe, European, Vegetarian Friendly","Vegetarian Friendly, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Creps Barcelona | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, Cafe, European, Vegetarian Friendly"
2,382403,g187497-d7814939,Hummus Barcelona Vegetarian Street Food,Barcelona,Spain,4.5,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, Brunch, Drinks","Reservations, Takeout, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Free Wifi, Accepts Credit Cards, Table Service","Hummus Barcelona Vegetarian Street Food | Barcelona, Spain | rating=4.5 | price=mid | tags=Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly"
3,381240,g187497-d6413303,SandwiChez,Barcelona,Spain,4.0,cheap,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Quick Bites, Cafe, Fast food","Vegetarian Friendly, Vegan Options","Breakfast, Lunch, Brunch",Unknown,"SandwiChez | Barcelona, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Quick Bites, Cafe, Fast food"
4,381215,g187497-d6367223,Bar Malasang,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Bar, Mediterranean, European","Vegetarian Friendly, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Bar Malasang | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, Bar, Mediterranean, European"
5,374459,g187497-d1083995,La Taverneta Nou,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, French, Mediterranean, European",Vegetarian Friendly,"Lunch, Dinner, Drinks, Brunch, Breakfast",Unknown,"La Taverneta Nou | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, French, Mediterranean, European"
6,379790,g187497-d3601243,Güell Tapas,Barcelona,Spain,4.5,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Mediterranean, European, Spanish","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch, After-hours","Reservations, Outdoor Seating, Seating, Television, Highchairs Available, Wheelchair Accessible, Serves Alcohol, Full Bar, Free Wifi, Accepts Credit Cards, Table Service, Live Music","Güell Tapas | Barcelona, Spain | rating=4.5 | price=mid | tags=Mid-range, Mediterranean, European, Spanish"
7,383168,g187497-d8737446,El Petit Firo,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Bar, Mediterranean, European","Vegetarian Friendly, Vegan Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"El Petit Firo | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, Bar, Mediterranean, European"
8,382900,g187497-d8462231,Al Taglio Bcn,Barcelona,Spain,4.5,cheap,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner, Brunch",Unknown,"Al Taglio Bcn | Barcelona, Spain | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
9,374450,g187497-d10836412,Zumito,Barcelona,Spain,4.5,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Vegetarian Friendly, Vegan Options, Gluten Free Options","Vegetarian Friendly, Vegan Options, Gluten Free Options","Breakfast, Lunch, Brunch, Dinner","Takeout, Reservations, Outdoor Seating, Seating, Wheelch

In [ ]:
query = "vegetarian restaurant in berlin"

parsed, filtered_df = parse_and_filter(
    query,
    df,
    min_results=20,
    verbose=True,
)

print("Unique top cities:")
display(filtered_df["city_filled"].value_counts(dropna=False).head(10))

preview_filtered_results(filtered_df, n=20)

Parsed query:
{
  "original_query": "vegetarian restaurant in berlin",
  "normalized_query": "vegetarian restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=berlin: 1033798 -> 0
Orașul a fost detectat, dar nu există în datasetul indexat. Returnăm subset gol ca să evităm recomandări greșite.
After location filter: 0
After price filter: 0
Metadata score filter > 0: 0 -> 0
Metadata filter prea strict. Păstrăm toate rezultatele, dar cu metadata_score calculat.
After metadata filter/scoring: 0
Unique top cities:


,count
city_filled,


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile


In [ ]:
query = "best cheap restaurant in northern ireland"

parsed, filtered_df = parse_and_filter(
    query,
    df,
    min_results=20,
    verbose=True,
)

print(json.dumps(parsed, indent=2, ensure_ascii=False))
preview_filtered_results(filtered_df, n=20)

Parsed query:
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
Country filter country=northern ireland: 1033798 -> 2549
After location filter: 2549
Price filter price_bucket=cheap: 2549 -> 517
After price filter: 517
Metadata score filter > 0: 517 -> 517
After metadata filter/scoring: 517
{
  "original_query": "best cheap restaurant in northern ireland",
  "normalized_query": "best cheap restaurant in northern ireland",
  "city": null,
  "country": "northern ireland",
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,604530,g7988500-d8492811,The Chip Yard,Whiteabbey,Northern Ireland,4.0,cheap,2.0,[price=cheap],"Cheap Eats, Fast food, British, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner",Unknown,"The Chip Yard | Whiteabbey, Northern Ireland | rating=4.0 | price=cheap | tags=Cheap Eats, Fast food, British, Vegetarian Friendly"
1,466728,g1010177-d9800362,McDonnell's Burger Bar,Belleek,Northern Ireland,4.5,cheap,2.0,[price=cheap],Cheap Eats,Unknown,Dinner,Unknown,"McDonnell's Burger Bar | Belleek, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats"
2,467037,g1019569-d2200644,Edfield Restaurant,Fivemiletown,Northern Ireland,4.5,cheap,2.0,[price=cheap],"Cheap Eats, Fast food, British, Diner","Vegetarian Friendly, Gluten Free Options","Breakfast, Lunch, Dinner",Unknown,"Edfield Restaurant | Fivemiletown, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Fast food, British, Diner"
3,467605,g1024119-d10163627,Peking Garden Chinese Take Away,Killyleagh,Northern Ireland,4.0,cheap,2.0,[price=cheap],"Cheap Eats, Chinese",Unknown,Unknown,Unknown,"Peking Garden Chinese Take Away | Killyleagh, Northern Ireland | rating=4.0 | price=cheap | tags=Cheap Eats, Chinese"
4,468119,g10429589-d6210861,The Rinka,Islandmagee,Northern Ireland,4.5,cheap,2.0,[price=cheap],"Cheap Eats, Dessert, Cafe",Unknown,Unknown,"Takeout, Outdoor Seating, Seating, Parking Available, Wheelchair Accessible","The Rinka | Islandmagee, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Dessert, Cafe"
5,596512,g635686-d5043735,Rolo's Fish and Chip Shop,Antrim,Northern Ireland,4.5,cheap,2.0,[price=cheap],"Cheap Eats, Fast food, British, Gluten Free Options",Gluten Free Options,"Lunch, Dinner",Unknown,"Rolo's Fish and Chip Shop | Antrim, Northern Ireland | rating=4.5 | price=cheap | tags=Cheap Eats, Fast food, British, Gluten Free Options"
6,596518,g635686-d7230168,Shunters Chip Shop and Smoke House,Antrim,Northern Ireland,3.5,cheap,2.0,[price=cheap],"Cheap Eats, Quick Bites, Fast food, Barbecue",Unknown,Lunch,"Delivery, Takeout, Seating","Shunters Chip Shop and Smoke House | Antrim, Northern Ireland | rating=3.5 | price=cheap | tags=Cheap Eats, Quick Bites, Fast food, Barbecue"
7,596519,g635686-d7273751,ASDA Cafe,Antrim,Northern Ireland,2.0,cheap,2.0,[price=cheap],"Cheap Eats, Quick Bites, Cafe, British",Unknown,Unknown,"Seating, Wheelchair Accessible","ASDA Cafe | Antrim, Northern Ireland | rating=2.0 | price=cheap | tags=Cheap Eats, Quick Bites, Cafe, British"
8,596520,g635686-d752812,Arcade Chip Shop,Antrim,Northern Ireland,4.0,cheap,2.0,[price=cheap],"Cheap Eats, Fast food, British",Unknown,"Lunch, Dinner",Takeout,"Arcade Chip Shop | Antrim, Northern Ireland | rating=4.0 | price=cheap | tags=Cheap Eats, Fast food, British"
9,596522,g635686-d752815,The Chippy,Antrim,Northern Ireland,3.0,cheap,2.0,[price=cheap],"Cheap Eats, Quick Bites, Fast food, British",Unknown,Lunch,"Delivery, Takeout, Wheelchair Accessible","The Chippy | Antrim, Northern Ireland | rating=3.0 | price=cheap | tags=Cheap Eats, Quick Bites, Fast food, British"


In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "vegan restaurant near berlin with outdoor seating",
    "tapas place in madrid",
    "family restaurant in london",
    "gluten free restaurant in paris",
]

batch_summary = []

for q in test_queries:
    print("=" * 120)
    print("QUERY:", q)

    parsed, subset = parse_and_filter(q, df, min_results=20, verbose=True)

    top_city = subset.iloc[0]["city_filled"] if len(subset) > 0 and "city_filled" in subset.columns else None
    top_price = subset.iloc[0]["price_bucket"] if len(subset) > 0 and "price_bucket" in subset.columns else None

    batch_summary.append({
        "query": q,
        "parsed_city": parsed.get("city"),
        "parsed_country": parsed.get("country"),
        "parsed_price_bucket": parsed.get("price_bucket"),
        "parsed_tags": parsed.get("tags"),
        "parsed_dietary": parsed.get("dietary"),
        "parsed_meals": parsed.get("matched_meals"),
        "parsed_features": parsed.get("matched_features"),
        "num_results_after_filter": len(subset),
        "top_city": top_city,
        "top_price": top_price,
    })

    display(preview_filtered_results(subset, n=5))

batch_summary_df = pd.DataFrame(batch_summary)
display(batch_summary_df)

QUERY: cheap italian restaurant in rome
Parsed query:
{
  "original_query": "cheap italian restaurant in rome",
  "normalized_query": "cheap italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Price filter price_bucket=cheap: 12603 -> 2755
After price filter: 2755
Metadata score filter > 0: 2755 -> 2755
After metadata filter/scoring: 2755


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,663015,g187791-d10043473,Forno Dolce Forno,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Bakeries, Italian",Unknown,Brunch,"Takeout, Wheelchair Accessible","Forno Dolce Forno | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Bakeries, Italian"
1,663012,g187791-d10042395,I Nuovi Gladiatori,Rome,Italy,3.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Takeout, Delivery, Reservations, Wheelchair Accessible","I Nuovi Gladiatori | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
2,675525,g187791-d9789769,Bar della Paglia,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian, European, Vegetarian Friendly",Vegetarian Friendly,Dinner,Unknown,"Bar della Paglia | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, European, Vegetarian Friendly"
3,675441,g187791-d9712051,Pasticceria Siciliana Roma,Rome,Italy,4.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Dessert, Italian, Cafe",Unknown,Breakfast,"Takeout, Wheelchair Accessible, Seating","Pasticceria Siciliana Roma | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Dessert, Italian, Cafe"
4,675471,g187791-d9735886,Harbour Cafe,Rome,Italy,5.0,cheap,4.0,"[price=cheap, tag=italian]","Cheap Eats, Italian",Unknown,"Brunch, Drinks, Breakfast, Lunch","Takeout, Outdoor Seating, Seating, Parking Available, Street Parking, Wheelchair Accessible, Serves Alcohol, Full Bar, Wine and Beer, Accepts Mastercard, Accepts Visa, Accepts Credit Cards","Harbour Cafe | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Italian"


QUERY: vegetarian brunch in barcelona
Parsed query:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
After price filter: 10283
Metadata score filter > 0: 10283 -> 3713
After metadata filter/scoring: 3713


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,379304,g187497-d2467666,Bar del Pi,Barcelona,Spain,4.0,cheap,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Bar, Cafe, Mediterranean",Vegetarian Friendly,"Lunch, Dinner, Brunch, Drinks",Unknown,"Bar del Pi | Barcelona, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Bar, Cafe, Mediterranean"
1,382885,g187497-d8448147,Creps Barcelona,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Cafe, European, Vegetarian Friendly","Vegetarian Friendly, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Creps Barcelona | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, Cafe, European, Vegetarian Friendly"
2,382403,g187497-d7814939,Hummus Barcelona Vegetarian Street Food,Barcelona,Spain,4.5,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options","Lunch, Dinner, Brunch, Drinks","Reservations, Takeout, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Free Wifi, Accepts Credit Cards, Table Service","Hummus Barcelona Vegetarian Street Food | Barcelona, Spain | rating=4.5 | price=mid | tags=Cheap Eats, Mediterranean, Middle Eastern, Vegetarian Friendly"
3,381240,g187497-d6413303,SandwiChez,Barcelona,Spain,4.0,cheap,3.0,"[diet=vegetarian friendly, meal=brunch]","Cheap Eats, Quick Bites, Cafe, Fast food","Vegetarian Friendly, Vegan Options","Breakfast, Lunch, Brunch",Unknown,"SandwiChez | Barcelona, Spain | rating=4.0 | price=cheap | tags=Cheap Eats, Quick Bites, Cafe, Fast food"
4,381215,g187497-d6367223,Bar Malasang,Barcelona,Spain,4.0,mid,3.0,"[diet=vegetarian friendly, meal=brunch]","Mid-range, Bar, Mediterranean, European","Vegetarian Friendly, Gluten Free Options","Breakfast, Lunch, Dinner, Brunch",Unknown,"Bar Malasang | Barcelona, Spain | rating=4.0 | price=mid | tags=Mid-range, Bar, Mediterranean, European"


QUERY: romantic dinner in paris
Parsed query:
{
  "original_query": "romantic dinner in paris",
  "normalized_query": "romantic dinner in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [
    "dinner"
  ],
  "matched_features": [
    "romantic"
  ]
}

Initial rows: 1033798
City filter city=paris: 1033798 -> 18126
After location filter: 18126
After price filter: 18126
Metadata score filter > 0: 18126 -> 9307
After metadata filter/scoring: 9307


...

QUERY: family restaurant in london
Parsed query:
{
  "original_query": "family restaurant in london",
  "normalized_query": "family restaurant in london",
  "city": null,
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
After location filter: 1033798
After price filter: 1033798
No metadata constraints found.
After metadata filter/scoring: 1033798


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,0,g10001637-d10002227,Le 147,Saint-Jouvent,France,4.0,cheap,0.0,[],"Cheap Eats, French",Unknown,"Lunch, Dinner","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French"
1,1,g10001637-d14975787,Le Saint Jouvent,Saint-Jouvent,France,4.0,cheap,0.0,[],Cheap Eats,Unknown,Unknown,Unknown,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats"
2,2,g10002858-d4586832,Au Bout du Pont,Rivarennes,France,5.0,cheap,0.0,[],"Cheap Eats, French, European",Unknown,"Dinner, Lunch, Drinks","Reservations, Seating, Table Service, Wheelchair Accessible","Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European"
3,3,g10002986-d3510044,Le Relais de Naiade,Lacelle,France,4.0,cheap,0.0,[],"Cheap Eats, French",Unknown,"Lunch, Dinner","Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible","Le Relais de Naiade | Lacelle, France | rating=4.0 | price=cheap | tags=Cheap Eats, French"
4,4,g10022428-d9767191,Relais Du MontSeigne,Saint-Laurent-de-Levezou,France,4.5,mid,0.0,[],"Mid-range, French",Unknown,"Lunch, Dinner","Reservations, Seating, Wheelchair Accessible, Table Service","Relais Du MontSeigne | Saint-Laurent-de-Levezou, France | rating=4.5 | price=mid | tags=Mid-range, French"


QUERY: gluten free restaurant in paris
Parsed query:
{
  "original_query": "gluten free restaurant in paris",
  "normalized_query": "gluten free restaurant in paris",
  "city": "paris",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "gluten free options"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=paris: 1033798 -> 18126
After location filter: 18126
After price filter: 18126
Metadata score filter > 0: 18126 -> 584
After metadata filter/scoring: 584


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,56537,g187147-d9996090,Gemini Les Halles,Paris,France,4.0,expensive,2.0,[diet=gluten free options],"Mid-range, Italian, Pizza, Mediterranean","Vegan Options, Gluten Free Options, Vegetarian Friendly",Unknown,Unknown,"Gemini Les Halles | Paris, France | rating=4.0 | price=expensive | tags=Mid-range, Italian, Pizza, Mediterranean"
1,38414,g187147-d10005308,Umami Matcha Cafe,Paris,France,4.0,expensive,2.0,[diet=gluten free options],"Mid-range, Japanese, Cafe, Asian","Vegetarian Friendly, Vegan Options, Gluten Free Options",Unknown,Unknown,"Umami Matcha Cafe | Paris, France | rating=4.0 | price=expensive | tags=Mid-range, Japanese, Cafe, Asian"
2,38427,g187147-d10028077,Juicy & Tasty,Paris,France,5.0,mid,2.0,[diet=gluten free options],"Cheap Eats, Mediterranean, Vegetarian Friendly, Vegan Options","Vegetarian Friendly, Vegan Options, Gluten Free Options",Unknown,Unknown,"Juicy & Tasty | Paris, France | rating=5.0 | price=mid | tags=Cheap Eats, Mediterranean, Vegetarian Friendly, Vegan Options"
3,38437,g187147-d10033776,Restaurant Simone Lemon,Paris,France,4.5,mid,2.0,[diet=gluten free options],"Cheap Eats, French, European, Deli","Vegetarian Friendly, Vegan Options, Gluten Free Options",Unknown,Unknown,"Restaurant Simone Lemon | Paris, France | rating=4.5 | price=mid | tags=Cheap Eats, French, European, Deli"
4,38440,g187147-d10035877,Health Inside,Paris,France,4.0,expensive,2.0,[diet=gluten free options],"Mid-range, International, Healthy, Vegetarian Friendly","Vegetarian Friendly, Vegan Options, Gluten Free Options",Unknown,Unknown,"Health Inside | Paris, France | rating=4.0 | price=expensive | tags=Mid-range, International, Healthy, Vegetarian Friendly"


,query,parsed_city,parsed_country,parsed_price_bucket,parsed_tags,parsed_dietary,parsed_meals,parsed_features,num_results_after_filter,top_city,top_price
0,cheap italian restaurant in rome,rome,None,cheap,[italian],[],[],[],2755,Rome,cheap
1,vegetarian brunch in barcelona,barcelona,None,None,[],[vegetarian friendly],[brunch],[],3713,Barcelona,cheap
2,romantic dinner in paris,paris,None,None,[],[],[dinner],[romantic],9307,Paris,expensive
3,family friendly seafood place in lisbon,lisbon,None,None,[seafood],[],[],[family friendly],286,Lisbon,expensive
4,coffee and breakfast in athens,athens,None,None,[coffee],[],[breakfast],[],580,Athens,expensive
5,expensive fine dining in milan,milan,None,expensive,[fine dining],[],[],[],2012,Milan,expensive
6,vegan restaurant near berlin with outdoor seating,berlin,None,None,[],[vegan options],[],"[outdoor seating, seating]",0,None,None
7,tapas place in madrid,madrid,None,None,[spanish],[],[],[],5305,Madrid,expensive
8,family restaurant in london,None,None,None,[],[],[],[],1033798,Saint-Jouvent,cheap
9,gluten free restaurant in paris,paris,None,None,[],[gluten free options],[],[],584,Paris,expensive


In [ ]:
city_diagnostic_queries = [
    ("cheap restaurant in rome", "rome"),
    ("restaurant in paris", "paris"),
    ("dinner near berlin", "berlin"),
    ("seafood in lisbon", "lisbon"),
    ("fine dining in milan", "milan"),
    ("tapas in madrid", "madrid"),
    ("breakfast in athens", "athens"),
    ("family restaurant in london", "london"),
]

city_diagnostics = []

for query, expected_city in city_diagnostic_queries:
    parsed = parse_user_query(query)

    city_diagnostics.append({
        "query": query,
        "expected_city": expected_city,
        "parsed_city": parsed.get("city"),
        "ok": parsed.get("city") == expected_city,
    })

city_diagnostics_df = pd.DataFrame(city_diagnostics)
display(city_diagnostics_df)

print("City parsing accuracy on diagnostic set:", city_diagnostics_df["ok"].mean())

,query,expected_city,parsed_city,ok
0,cheap restaurant in rome,rome,rome,True
1,restaurant in paris,paris,paris,True
2,dinner near berlin,berlin,berlin,True
3,seafood in lisbon,lisbon,lisbon,True
4,fine dining in milan,milan,milan,True
5,tapas in madrid,madrid,madrid,True
6,breakfast in athens,athens,athens,True
7,family restaurant in london,london,None,False


City parsing accuracy on diagnostic set: 0.875


In [ ]:
price_diagnostic_queries = [
    ("cheap restaurant in rome", "cheap"),
    ("budget pizza in rome", "cheap"),
    ("affordable restaurant in paris", "cheap"),
    ("mid range restaurant in madrid", "mid"),
    ("moderate restaurant in lisbon", "mid"),
    ("fine dining in milan", "expensive"),
    ("luxury restaurant in paris", "expensive"),
    ("fancy dinner in london", "expensive"),
]

price_diagnostics = []

for query, expected_price in price_diagnostic_queries:
    parsed = parse_user_query(query)

    price_diagnostics.append({
        "query": query,
        "expected_price": expected_price,
        "parsed_price": parsed.get("price_bucket"),
        "ok": parsed.get("price_bucket") == expected_price,
    })

price_diagnostics_df = pd.DataFrame(price_diagnostics)
display(price_diagnostics_df)

print("Price parsing accuracy on diagnostic set:", price_diagnostics_df["ok"].mean())

,query,expected_price,parsed_price,ok
0,cheap restaurant in rome,cheap,cheap,True
1,budget pizza in rome,cheap,cheap,True
2,affordable restaurant in paris,cheap,cheap,True
3,mid range restaurant in madrid,mid,mid,True
4,moderate restaurant in lisbon,mid,mid,True
5,fine dining in milan,expensive,expensive,True
6,luxury restaurant in paris,expensive,expensive,True
7,fancy dinner in london,expensive,expensive,True


Price parsing accuracy on diagnostic set: 1.0


In [ ]:
parser_config = {
    "price_keywords": PRICE_KEYWORDS,
    "tag_synonyms": TAG_SYNONYMS,
    "dietary_synonyms": DIETARY_SYNONYMS,
    "meal_synonyms": MEAL_SYNONYMS,
    "feature_synonyms": FEATURE_SYNONYMS,
    "country_aliases": COUNTRY_ALIASES,
    "city_aliases": CITY_ALIASES,
    "bad_city_terms": sorted(BAD_CITY_TERMS),
    "num_countries": len(country_vocab),
    "num_cities": len(city_vocab),
    "num_tags": len(tag_vocab),
    "num_meals": len(meal_vocab),
    "num_features": len(feature_vocab),
    "num_diets": len(diet_vocab),
}

PARSER_CONFIG_PATH = OUTPUT_DIR / "query_parser_config.json"

with open(PARSER_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(parser_config, f, indent=2, ensure_ascii=False)

print("Saved parser config:", PARSER_CONFIG_PATH)

Saved parser config: /content/drive/MyDrive/tablewise/artifacts_new/query_parser/query_parser_config.json


In [ ]:
query_examples_df = pd.DataFrame([
    parse_user_query(q) for q in test_queries
])

QUERY_EXAMPLES_PATH = OUTPUT_DIR / "query_examples.csv"

query_examples_df.to_csv(QUERY_EXAMPLES_PATH, index=False)

print("Saved query examples:", QUERY_EXAMPLES_PATH)

display(query_examples_df)

Saved query examples: /content/drive/MyDrive/tablewise/artifacts_new/query_parser/query_examples.csv


,original_query,normalized_query,city,country,price_bucket,tags,dietary,matched_meals,matched_features
0,cheap italian restaurant in rome,cheap italian restaurant in rome,rome,None,cheap,[italian],[],[],[]
1,vegetarian brunch in barcelona,vegetarian brunch in barcelona,barcelona,None,None,[],[vegetarian friendly],[brunch],[]
2,romantic dinner in paris,romantic dinner in paris,paris,None,None,[],[],[dinner],[romantic]
3,family friendly seafood place in lisbon,family friendly seafood place in lisbon,lisbon,None,None,[seafood],[],[],[family friendly]
4,coffee and breakfast in athens,coffee and breakfast in athens,athens,None,None,[coffee],[],[breakfast],[]
5,expensive fine dining in milan,expensive fine dining in milan,milan,None,expensive,[fine dining],[],[],[]
6,vegan restaurant near berlin with outdoor seating,vegan restaurant near berlin with outdoor seating,berlin,None,None,[],[vegan options],[],"[outdoor seating, seating]"
7,tapas place in madrid,tapas place in madrid,madrid,None,None,[spanish],[],[],[]
8,family restaurant in london,family restaurant in london,None,None,None,[],[],[],[]
9,gluten free restaurant in paris,gluten free restaurant in paris,paris,None,None,[],[gluten free options],[],[]


In [ ]:
vocab_artifact = {
    "country_vocab": country_vocab,
    "city_vocab_sample": city_vocab[:5000],
    "tag_vocab_sample": tag_vocab[:5000],
    "meal_vocab": meal_vocab,
    "feature_vocab_sample": feature_vocab[:5000],
    "diet_vocab": diet_vocab,
}

VOCAB_PATH = OUTPUT_DIR / "query_parser_vocab.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_artifact, f, indent=2, ensure_ascii=False)

print("Saved vocab artifact:", VOCAB_PATH)

Saved vocab artifact: /content/drive/MyDrive/tablewise/artifacts_new/query_parser/query_parser_vocab.json


In [ ]:
BATCH_SUMMARY_PATH = OUTPUT_DIR / "query_parser_batch_summary.csv"

batch_summary_df.to_csv(BATCH_SUMMARY_PATH, index=False)

print("Saved batch summary:", BATCH_SUMMARY_PATH)
display(batch_summary_df)

Saved batch summary: /content/drive/MyDrive/tablewise/artifacts_new/query_parser/query_parser_batch_summary.csv


,query,parsed_city,parsed_country,parsed_price_bucket,parsed_tags,parsed_dietary,parsed_meals,parsed_features,num_results_after_filter,top_city,top_price
0,cheap italian restaurant in rome,rome,None,cheap,[italian],[],[],[],2755,Rome,cheap
1,vegetarian brunch in barcelona,barcelona,None,None,[],[vegetarian friendly],[brunch],[],3713,Barcelona,cheap
2,romantic dinner in paris,paris,None,None,[],[],[dinner],[romantic],9307,Paris,expensive
3,family friendly seafood place in lisbon,lisbon,None,None,[seafood],[],[],[family friendly],286,Lisbon,expensive
4,coffee and breakfast in athens,athens,None,None,[coffee],[],[breakfast],[],580,Athens,expensive
5,expensive fine dining in milan,milan,None,expensive,[fine dining],[],[],[],2012,Milan,expensive
6,vegan restaurant near berlin with outdoor seating,berlin,None,None,[],[vegan options],[],"[outdoor seating, seating]",0,None,None
7,tapas place in madrid,madrid,None,None,[spanish],[],[],[],5305,Madrid,expensive
8,family restaurant in london,None,None,None,[],[],[],[],1033798,Saint-Jouvent,cheap
9,gluten free restaurant in paris,paris,None,None,[],[gluten free options],[],[],584,Paris,expensive


In [ ]:
def tablewise_parse_query(query):
    return parse_user_query(query)


def tablewise_filter_candidates(query, mapping_df, min_results=20, verbose=False):
    parsed, filtered = parse_and_filter(
        query=query,
        df_in=mapping_df,
        min_results=min_results,
        verbose=verbose,
    )

    return parsed, filtered

In [ ]:
query = "cheap vegetarian italian restaurant in rome"

parsed, candidates = tablewise_filter_candidates(
    query,
    df,
    min_results=20,
    verbose=True,
)

print(json.dumps(parsed, indent=2, ensure_ascii=False))

preview_filtered_results(candidates, n=20)

Parsed query:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Price filter price_bucket=cheap: 12603 -> 2755
After price filter: 2755
Metadata score filter > 0: 2755 -> 2755
After metadata filter/scoring: 2755
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}


,faiss_idx,restaurant_id,name,city_filled,country,rating,price_bucket,metadata_score,metadata_match_reasons,top_tags_text,special_diets_text,meals_text,features_text,short_profile
0,663012,g187791-d10042395,I Nuovi Gladiatori,Rome,Italy,3.0,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Takeout, Delivery, Reservations, Wheelchair Accessible","I Nuovi Gladiatori | Rome, Italy | rating=3.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
1,675585,g187791-d9998177,Il Maghetto,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,Dinner,Unknown,"Il Maghetto | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
2,662987,g187791-d10021003,Pizza Gegè,Rome,Italy,4.0,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Vegetarian Friendly",Vegetarian Friendly,Unknown,"Takeout, Serves Alcohol","Pizza Gegè | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, Pizza, Vegetarian Friendly"
3,675577,g187791-d9977682,Bar Farnese,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Cafe, Vegetarian Friendly",Vegetarian Friendly,Breakfast,Unknown,"Bar Farnese | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Cafe, Vegetarian Friendly"
4,675584,g187791-d9996227,Pizzeria Vecchio Borgo,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Fast food, Romana","Vegetarian Friendly, Vegan Options","Lunch, Dinner, Brunch, Drinks",Unknown,"Pizzeria Vecchio Borgo | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Fast food, Romana"
5,662988,g187791-d10021211,Bistrot Balduina,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Seafood, Mediterranean","Vegetarian Friendly, Vegan Options","Breakfast, Dinner, Lunch",Unknown,"Bistrot Balduina | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Seafood, Mediterranean"
6,675525,g187791-d9789769,Bar della Paglia,Rome,Italy,4.0,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, European, Vegetarian Friendly",Vegetarian Friendly,Dinner,Unknown,"Bar della Paglia | Rome, Italy | rating=4.0 | price=cheap | tags=Cheap Eats, Italian, European, Vegetarian Friendly"
7,675528,g187791-d9791601,Gelateria Cinque,Rome,Italy,5.0,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Dessert, Italian, Vegetarian Friendly","Gluten Free Options, Vegetarian Friendly, Vegan Options",Unknown,"Takeout, Wheelchair Accessible, Accepts Mastercard, Accepts Visa, Digital Payments, Accepts Credit Cards, Free Wifi","Gelateria Cinque | Rome, Italy | rating=5.0 | price=cheap | tags=Cheap Eats, Dessert, Italian, Vegetarian Friendly"
8,673366,g187791-d6959710,Pinseria La Fraschetta 2,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Pizza, Mediterranean",Vegetarian Friendly,Dinner,Unknown,"Pinseria La Fraschetta 2 | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Pizza, Mediterranean"
9,673367,g187791-d6959770,Piadineria artigianale I pazzi sognanti,Rome,Italy,4.5,cheap,6.0,"[diet=vegetarian friendly, price=cheap, tag=italian]","Cheap Eats, Italian, Fast food, Vegetarian Friendly",Vegetarian Friendly,"Lunch, Dinner, After-hours",Unknown,"Piadineria artigianale I pazzi sognanti | Rome, Italy | rating=4.5 | price=cheap | tags=Cheap Eats, Italian, Fast food, Vegetarian Friendly"
